1. Setup and Config

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy pyarrow     # install the tools (quietly)

import os                                    # files and folders
import numpy as np                           # maths helpers
import pandas as pd                          # tables
import duckdb                                # runs SQL on the data files
from google.colab import userdata            # Colab's secret store (where your token lives)
from huggingface_hub import HfApi, hf_hub_download   # list and download files from Hugging Face

HF_TOKEN = userdata.get("HF_TOKEN")          # read the secret token (never printed or typed here)
REPO = "FlyRank/internship-warehouse"        # the warehouse dataset

# ---------- CONFIG: the only block you change when new data arrives ----------
PRIOR_MONTHS  = ["2026-02"]                  # months that make up the "prior" window (features come from here)
FUTURE_MONTHS = ["2026-03"]                  # months that make up the "future" window (the outcome)
MIN_PRIOR_IMPR = 100                         # ignore pages with fewer prior impressions (too noisy)
DECLINE_RATIO  = 0.8                         # declined = future daily impressions below 80% of prior
CLIENT_DIP     = -0.10                       # a client "dipped" if its typical page fell 10% or more
MIN_CLIENT_PAGES = 20                        # need at least 20 eligible pages to trust a client's typical change
# June 2026 stays sealed: never put "2026-06" in either list while developing

2. The Reusable Window Builder

In [2]:
api = HfApi(token=HF_TOKEN)                                              # connect to Hugging Face
ALL_FILES = api.list_repo_files(REPO, repo_type="dataset")               # list every file once

def fetch_months(months):                                                # download the daily-table files for some months
    paths = []                                                           # we will collect the local file paths here
    for m in months:                                                     # one month at a time
        wanted = [f for f in ALL_FILES
                  if f.startswith(f"fact_content_daily_performance/month={m}/")]   # files belonging to that month
        for f in wanted:                                                 # download each file (cached after the first time)
            paths.append(hf_hub_download(repo_id=REPO, repo_type="dataset", filename=f, token=HF_TOKEN))
    return paths                                                         # the list of local file paths

def sql_list(items):                                                     # turn ["a","b"] into 'a', 'b' for use in SQL
    return ", ".join("'" + x + "'" for x in items)

def build_windows(prior_months, future_months):                          # months in, one table of pages out
    months = prior_months + future_months                                # all the months we need
    paths = fetch_months(months)                                         # download them
    win_case = f"CASE WHEN month IN ({sql_list(prior_months)}) THEN 'prior' ELSE 'future' END"   # tag each row prior or future

    days = duckdb.sql(f"""
        SELECT {win_case} AS win, COUNT(DISTINCT report_date) AS n_days  -- calendar days in each window
        FROM read_parquet({paths}) WHERE month IN ({sql_list(months)}) GROUP BY win
    """).df().set_index("win")["n_days"]                                 # a small lookup: days per window

    long = duckdb.sql(f"""
        SELECT client_hash_id, content_hash_id, {win_case} AS win,       -- one row per page and window
               CAST(SUM(gsc_impressions) AS DOUBLE) AS impr              -- total impressions in that window
        FROM read_parquet({paths})
        WHERE gsc_data_available IS TRUE AND month IN ({sql_list(months)})   -- only rows with real search data
        GROUP BY client_hash_id, content_hash_id, win
    """).df()                                                            # run the query and get a table

    page = long.pivot_table(index=["client_hash_id", "content_hash_id"],
                            columns="win", values="impr", aggfunc="sum").fillna(0).reset_index()  # one row per page, with prior and future columns
    page["prior_daily"]  = page["prior"]  / days["prior"]                # average impressions per day, prior window
    page["future_daily"] = page["future"] / days["future"]               # average impressions per day, future window

    client = page.groupby("client_hash_id")[["prior_daily", "future_daily"]].sum()   # each client's total daily traffic
    client["client_total_change"] = (client["future_daily"] / client["prior_daily"]).replace([np.inf, -np.inf], np.nan) - 1   # its overall change
    page = page.merge(client[["client_total_change"]], left_on="client_hash_id", right_index=True, how="left")  # attach to each page

    page = page[page["prior"] >= MIN_PRIOR_IMPR].copy()                  # keep pages with enough prior impressions
    page["page_change"] = page["future_daily"] / page["prior_daily"] - 1 # this page's change (-0.25 means 25% fewer per day)
    grp = page.groupby("client_hash_id")["page_change"]                  # group pages by client
    page["client_median_change"] = grp.transform("median")               # the typical page's change in this client
    page.loc[grp.transform("size") < MIN_CLIENT_PAGES, "client_median_change"] = np.nan   # too few pages: do not trust it
    page["relative_change"] = page["page_change"] - page["client_median_change"]   # this page versus its client's typical page
    return page                                                          # the finished page table

3. Run It And Sort The Declines

In [3]:
page = build_windows(PRIOR_MONTHS, FUTURE_MONTHS)                        # build the table for the configured months

page["declined"] = (page["future_daily"] < DECLINE_RATIO * page["prior_daily"]).astype(int)   # 1 if daily impressions fell below 80%
page["client_dipped"] = page["client_median_change"] <= CLIENT_DIP       # True if the client's typical page also fell
page["bucket"] = np.select(
    [page["declined"].eq(0),                                             # not declined
     page["declined"].eq(1) & page["client_dipped"],                     # declined and the client dipped too
     page["declined"].eq(1) & page["client_median_change"].notna()],     # declined while the client held steady
    ["not declined", "declined with client-wide dip", "declined, client steady"],
    default="declined, client unknown")                                  # declined, but too few pages to judge

print(f"Eligible pages: {len(page):,} | clients: {page['client_hash_id'].nunique()} | declined: {page['declined'].mean():.1%}")  # size and base rate
print(page["bucket"].value_counts().to_string())                         # how many pages fall in each bucket
print(page.groupby("bucket")["relative_change"].median().round(3).to_string())   # typical change versus the client's typical page

os.makedirs("work/outputs", exist_ok=True)                               # make the output folder in this session
page.to_parquet(f"work/outputs/windows_{PRIOR_MONTHS[0]}_{FUTURE_MONTHS[-1]}.parquet", index=False)  # cache the table (a data file: stays out of git)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Eligible pages: 80,322 | clients: 37 | declined: 26.7%
bucket
not declined                     58840
declined, client steady          17837
declined with client-wide dip     3609
declined, client unknown            36
bucket
declined with client-wide dip    0.000
declined, client steady         -0.501
declined, client unknown           NaN
not declined                     0.178


4. Did The Client Dip The Same Way Last Year?

In [4]:
last = build_windows(["2025-02"], ["2025-03"])                           # the same Feb-to-Mar change, one year earlier
ly = last.groupby("client_hash_id")["client_total_change"].first().rename("client_change_last_year")  # each client's change that year
page = page.merge(ly, left_on="client_hash_id", right_index=True, how="left")   # attach it to this year's pages
page["usual_dip"] = page["client_change_last_year"] <= CLIENT_DIP        # True if the client also dipped Feb-to-Mar last year
print("Pages with last-year information:", page["client_change_last_year"].notna().sum())   # how many pages we could compare
print(page.groupby("bucket")["usual_dip"].agg(["size", "sum"]).to_string())   # per bucket: pages, and how many have a "usual dip" client

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 1.45MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Pages with last-year information: 20511
                                size  sum
bucket                                   
declined with client-wide dip   3609    0
declined, client steady        17837   79
declined, client unknown          36    0
not declined                   58840  764


5. Build the 14-day vs 14-day numbers

In [5]:
# --- Settings for the baseline (write these down before looking at results, then freeze them) ---
DROP_THRESHOLD = 0.20     # flag a page if its impressions fell by 20% or more (mirrors the 80% label line)
MIN_BEFORE14 = 100        # the page needs at least 100 impressions in the earlier 14 days
TEST_FRACTION = 0.2       # share of clients held out for testing
SEED = 42                 # fixed random seed so the split is repeatable

assert max(PRIOR_MONTHS) < min(FUTURE_MONTHS), "prior months must come before future months"   # features may never come from the outcome window

prior_paths = fetch_months(PRIOR_MONTHS)                      # local file paths for the prior months (already downloaded)

# For each page: impressions in the last 14 days of the prior window, and in the 14 days before that
rec = duckdb.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS last_day                    -- the decision day: the last day of the prior window
        FROM read_parquet({prior_paths}) WHERE month IN ({sql_list(PRIOR_MONTHS)})
    )
    SELECT f.client_hash_id, f.content_hash_id,
           CAST(SUM(f.gsc_impressions) FILTER (WHERE f.report_date > b.last_day - INTERVAL 14 DAY) AS DOUBLE) AS recent14,   -- last 14 days
           CAST(SUM(f.gsc_impressions) FILTER (WHERE f.report_date <= b.last_day - INTERVAL 14 DAY
                                                 AND f.report_date >  b.last_day - INTERVAL 28 DAY) AS DOUBLE) AS before14   -- the 14 days before
    FROM read_parquet({prior_paths}) f CROSS JOIN bounds b
    WHERE f.gsc_data_available IS TRUE AND f.month IN ({sql_list(PRIOR_MONTHS)})   -- only rows with real search data
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()                                                      # run the query and get a table

base = page.merge(rec, on=["client_hash_id", "content_hash_id"], how="left")   # attach the two numbers to each page
print(f"Pages: {len(base):,} | with both 14-day numbers: {base[['recent14','before14']].notna().all(axis=1).sum():,}")  # how many pages have both

Pages: 80,322 | with both 14-day numbers: 70,927


6. The rule, the held-out split and the scores

In [6]:
from sklearn.model_selection import GroupShuffleSplit          # splits data keeping each client's pages together
import json                                                    # to save a small results file

# Pages the rule can judge: enough earlier impressions, and a recent number that exists
elig = base[(base["before14"] >= MIN_BEFORE14) & base["recent14"].notna()].copy()   # keep only judgeable pages

elig["rel_drop"] = (elig["before14"] - elig["recent14"]) / elig["before14"]         # how much impressions fell (0.30 = down 30%)
elig["flag"] = elig["rel_drop"] >= DROP_THRESHOLD              # the rule: flag if the fall is 20% or more

# Rank by size of fall, then by prior daily impressions (the tie policy), highest first
ranked_all = elig.sort_values(["rel_drop", "prior_daily"], ascending=[False, False]).reset_index(drop=True)

# Hold out whole clients, so a client's pages are never in both groups
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=SEED)  # 80% / 20% by client
_, test_idx = next(splitter.split(elig, groups=elig["client_hash_id"]))              # positions of the test pages
test_clients = set(elig.iloc[test_idx]["client_hash_id"])      # which clients are held out
ranked_test = ranked_all[ranked_all["client_hash_id"].isin(test_clients)].reset_index(drop=True)  # the ranking restricted to them

def summarize(ranked, label):
    base_rate = ranked["declined"].mean()                      # share of judgeable pages that really declined
    out = {"group": label, "pages": len(ranked), "clients": ranked["client_hash_id"].nunique(),
           "base_rate": round(float(base_rate), 3), "flagged": int(ranked["flag"].sum()),
           "flagged_precision": round(float(ranked.loc[ranked["flag"], "declined"].mean()), 3)}   # share of flagged pages that declined
    for k in (50, 100, 500):                                   # precision in the top 50, 100 and 500
        if len(ranked) >= k:                                   # only if the list is long enough
            out[f"precision_at_{k}"] = round(float(ranked["declined"].head(k).mean()), 3)
    return out

rows = [summarize(ranked_all, "all judgeable pages"), summarize(ranked_test, "held-out clients")]   # both views
print(pd.DataFrame(rows).T.to_string(header=False))            # show them side by side

os.makedirs("work/outputs", exist_ok=True)                     # make sure the output folder exists
json.dump({"settings": {"drop_threshold": DROP_THRESHOLD, "min_before14": MIN_BEFORE14, "seed": SEED,
                        "prior": PRIOR_MONTHS, "future": FUTURE_MONTHS},
           "results": rows}, open("work/outputs/prior_window_baseline_metrics.json", "w"), indent=2)   # save the receipts

group              all judgeable pages  held-out clients
pages                            57841               958
clients                             27                 6
base_rate                        0.287             0.356
flagged                          12838               384
flagged_precision                0.546             0.536
precision_at_50                   0.94               1.0
precision_at_100                  0.92              0.92
precision_at_500                 0.888             0.474


7. Look at the misses (the review step)

In [7]:
# The pages the rule ranked highest that did NOT decline: the false alarms to read closely
miss = ranked_test[~ranked_test["declined"].astype(bool)].head(10)    # top 10 highest-ranked non-decliners
cols = ["rel_drop", "before14", "recent14", "prior_daily", "future_daily", "client_median_change"]   # facts to inspect
print(miss[cols].round(2).to_string(index=False))              # show them as plain text

 rel_drop  before14  recent14  prior_daily  future_daily  client_median_change
     0.86     154.0      21.0         6.25          5.84                 -0.09
     0.84     172.0      28.0         7.14          6.26                  0.35
     0.82     146.0      26.0         6.14         11.52                  0.35
     0.82     381.0      68.0        16.04         14.55                  0.35
     0.77     172.0      39.0         7.54          9.45                  0.35
     0.75     528.0     130.0        23.50         24.03                  0.35
     0.75     156.0      39.0         6.96          6.35                 -0.09
     0.75     112.0      28.0         5.00          4.84                 -0.09
     0.71     138.0      40.0         6.36          9.26                 -0.09
     0.71     372.0     109.0        17.18         17.23                 -0.09


8. The grouped-fold cell

In [8]:
from sklearn.model_selection import GroupKFold           # splits data into folds, keeping each client's pages together
import json                                              # to save a small results file

folds = GroupKFold(n_splits=5)                           # 5 folds: every client is held out exactly once
rows = []                                                # we will collect one row of results per fold here

for i, (_, idx) in enumerate(folds.split(elig, groups=elig["client_hash_id"]), start=1):   # go through the folds one by one
    part = elig.iloc[idx]                                # the pages of the clients held out in this fold
    ranked = part.sort_values(["rel_drop", "prior_daily"], ascending=[False, False]).reset_index(drop=True)  # same ranking and tie policy as version 1
    res = summarize(ranked, f"fold {i}")                 # reuse the summary function from Cell F (same metrics)
    res["lift_flagged"] = round(res["flagged_precision"] / res["base_rate"], 2)   # flagged precision compared with the base rate
    rows.append(res)                                     # remember this fold's results

folds_table = pd.DataFrame(rows).set_index("group")      # one row per fold
print(folds_table.to_string())                           # show every fold, not only the average

num = folds_table.drop(columns=[]).select_dtypes("number")           # the numeric columns only
print("\\nmean:\\n", num.mean().round(3).to_string())      # the average across folds
print("\\nmin and max:\\n", pd.DataFrame({"min": num.min(), "max": num.max()}).round(3).to_string())   # the spread across folds

os.makedirs("work/outputs", exist_ok=True)               # make sure the output folder exists
json.dump(rows, open("work/outputs/prior_window_baseline_folds.json", "w"), indent=2)   # save the receipts

        pages  clients  base_rate  flagged  flagged_precision  precision_at_50  precision_at_100  precision_at_500  lift_flagged
group                                                                                                                           
fold 1  14787        1      0.261     2297              0.385             0.78              0.79             0.576          1.48
fold 2  12756        1      0.300     3275              0.465             1.00              0.97             0.784          1.55
fold 3  10099        7      0.238     1525              0.562             0.96              0.94             0.756          2.36
fold 4  10100        7      0.432     3358              0.853             0.96              0.98             0.976          1.97
fold 5  10099       11      0.212     2383              0.371             0.68              0.63             0.462          1.75
\nmean:\n pages                11568.200
clients                  5.400
base_rate                